# toc

> Generate structured Table of Contents from video xscripts via LLM

In [ ]:
#| default_exp toc

## Design

LLM returns `[{title, start}, ...]`. App-side adds `path` (sequential) and `end` (next start or duration).

**Pipeline:**
1. `_build_toc_prompt(segments, meta)` — build prompt from xscript segments
2. `_call_llm(prompt)` — call OpenAI gpt-5.4, return raw `[{title, start}]`
3. `_normalize_sections(raw, duration)` — add path/end, sort, dedup, validate
4. Write `toc.json` to cache dir

**toc.json format:**
```json
{"sections": [{"path": "1", "title": "Introduction", "start": 0, "end": 750}]}
```

In [ ]:
#| export
import json
from pathlib import Path

In [ ]:
#| export
def _normalize_sections(raw: list[dict], # [{title, start}, ...] from LLM
                        duration: int # Video duration in seconds
                       ) -> list[dict]: # [{path, title, start, end}, ...]
    "Add path/end, sort by start, dedup, validate. Raise on broken coverage."
    if not raw:
        raise ValueError("No sections returned from LLM")
    # Sort by start ascending
    sections = sorted(raw, key=lambda s: s['start'])
    # Remove duplicate starts (keep first)
    seen = set()
    deduped = []
    for s in sections:
        if s['start'] not in seen:
            seen.add(s['start'])
            deduped.append(s)
    sections = deduped
    # Fix first section start to 0
    sections[0] = {**sections[0], 'start': 0}
    # Add path and end
    result = []
    for i, s in enumerate(sections):
        end = sections[i+1]['start'] if i+1 < len(sections) else duration
        result.append({
            'path': str(i+1),
            'title': s['title'],
            'start': s['start'],
            'end': end,
        })
    return result

In [ ]:
#| export
def _build_toc_prompt(segments: list[dict], # [{start, end, text}, ...] from parse_xscript
                      meta: dict # meta.json content
                     ) -> str: # Prompt for LLM
    "Build a prompt that asks the LLM to identify topic transitions and return section titles with start times."
    lines = []
    for s in segments:
        mm = int(s['start'] // 60)
        ss = int(s['start'] % 60)
        lines.append(f"[{mm:02d}:{ss:02d}] {s['text']}")
    transcript = '\n'.join(lines)

    title = meta.get('title', '')
    channel = meta.get('channel', '')
    desc = meta.get('description', '')

    return f"""You are a structural editor for YouTube video transcripts.

Video info:
- Title: {title}
- Channel: {channel}
- Description: {desc}

Transcript:
{transcript}

Task:
Read the transcript and identify topic transitions. For each section, provide:
- title: concise English section title
- start: start time in integer seconds

Aim for 7-15 sections. Be faithful to transcript timestamps."""

_TOC_SCHEMA = {
    "type": "object",
    "properties": {
        "sections": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "title": {"type": "string"},
                    "start": {"type": "integer"},
                },
                "required": ["title", "start"],
            },
        },
    },
    "required": ["sections"],
}

def _call_llm(prompt: str # Full prompt
             ) -> list[dict]: # [{title, start}, ...]
    "Call OpenAI gpt-5.4 with structured output, return raw section list."
    import openai
    client = openai.OpenAI()
    response = client.chat.completions.create(
        model='gpt-5.4',
        response_format={
            "type": "json_schema",
            "json_schema": {"name": "generate_toc", "schema": _TOC_SCHEMA},
        },
        messages=[{"role": "user", "content": prompt}],
    )
    result = json.loads(response.choices[0].message.content)
    return result['sections']

## Tests

In [ ]:
# Test 1: basic — adds path and computes end from next start; last end = duration
raw = [{'title': 'Intro', 'start': 0}, {'title': 'Main', 'start': 300}, {'title': 'Outro', 'start': 600}]
secs = _normalize_sections(raw, duration=900)
assert len(secs) == 3
assert secs[0] == {'path': '1', 'title': 'Intro', 'start': 0, 'end': 300}
assert secs[1] == {'path': '2', 'title': 'Main', 'start': 300, 'end': 600}
assert secs[2] == {'path': '3', 'title': 'Outro', 'start': 600, 'end': 900}
print('ok')

In [ ]:
# Test 2: sorts by start ascending
raw = [{'title': 'B', 'start': 300}, {'title': 'A', 'start': 0}]
secs = _normalize_sections(raw, duration=600)
assert secs[0]['title'] == 'A'
assert secs[1]['title'] == 'B'
print('ok')

In [ ]:
# Test 3: removes duplicate starts (keeps first occurrence after sort)
raw = [{'title': 'A', 'start': 0}, {'title': 'A-dup', 'start': 0}, {'title': 'B', 'start': 300}]
secs = _normalize_sections(raw, duration=600)
assert len(secs) == 2
assert secs[0]['title'] == 'A'
assert secs[1]['title'] == 'B'
print('ok')

In [ ]:
# Test 4: fixes first section start to 0 if not already
raw = [{'title': 'Late start', 'start': 30}, {'title': 'Next', 'start': 300}]
secs = _normalize_sections(raw, duration=600)
assert secs[0]['start'] == 0
assert secs[0]['end'] == 300
print('ok')

In [ ]:
# Test 5: empty input raises ValueError
try:
    _normalize_sections([], duration=600)
    assert False, 'should have raised'
except ValueError:
    pass
print('ok')

In [ ]:
# Test 6: _build_toc_prompt includes transcript and meta context
segments = [
    {'start': 0.0, 'end': 5.0, 'text': 'hello world'},
    {'start': 5.0, 'end': 10.0, 'text': 'second segment'},
]
meta = {'title': 'Test Video', 'channel': 'Ch', 'description': 'A test video'}
prompt = _build_toc_prompt(segments, meta)
assert 'hello world' in prompt
assert 'second segment' in prompt
assert 'Test Video' in prompt
assert '[00:00]' in prompt  # timestamps formatted
print('ok')

## CLI

In [ ]:
#| export
import sys
from fastcore.script import call_parse
from yttoc.core import format_header, format_toc_line
from yttoc.fetch import _DEFAULT_ROOT, _update_last_used, _glob_srt
from yttoc.xscript import parse_xscript

def generate_toc(video_id: str, # Exact video_id
                 root: Path = None, # Root cache directory
                 refresh: bool = False, # Delete cached toc/summaries and regenerate
                ) -> list[dict]: # Normalized sections
    "Generate toc.json for a cached video. Returns sections list."
    root = root or _DEFAULT_ROOT
    d = root / video_id
    meta_path = d / 'meta.json'
    toc_path = d / 'toc.json'
    srt_files = _glob_srt(d)
    if not (meta_path.exists() and srt_files):
        raise SystemExit(f"Not cached: {video_id}")

    if refresh:
        if toc_path.exists(): toc_path.unlink()
        sum_path = d / 'summaries.json'
        if sum_path.exists():
            sum_path.unlink()
            print('Invalidated summaries.json (depends on toc)', file=sys.stderr)

    # Return cached toc if exists
    if toc_path.exists():
        return json.loads(toc_path.read_text(encoding='utf-8'))['sections']

    meta = json.loads(meta_path.read_text(encoding='utf-8'))
    segments = parse_xscript(srt_files[0])
    prompt = _build_toc_prompt(segments, meta)
    raw = _call_llm(prompt)
    sections = _normalize_sections(raw, meta.get('duration', 0))

    toc_path.write_text(
        json.dumps({'sections': sections}, indent=2, ensure_ascii=False),
        encoding='utf-8')
    _update_last_used(meta_path)
    return sections

@call_parse
def yttoc_toc(video_id: str, # Exact video_id
              root: str = None, # Root cache directory
              refresh: bool = False, # Regenerate toc (and invalidate summaries)
             ):
    "Generate and display Table of Contents for a cached video."
    root = Path(root) if root else _DEFAULT_ROOT
    d = root / video_id
    meta_path = d / 'meta.json'
    if not meta_path.exists():
        raise SystemExit(f"Not cached: {video_id}")

    meta = json.loads(meta_path.read_text(encoding='utf-8'))
    sections = generate_toc(video_id, root, refresh=refresh)

    print(format_header(meta))
    print()
    url = meta.get('webpage_url', '')
    for s in sections:
        print(format_toc_line(s, url))

In [ ]:
# Test 7: generate_toc returns cached toc.json without calling LLM
from tempfile import TemporaryDirectory
import io, contextlib

with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'VID1'; v.mkdir()
    (v / 'captions.en.srt').write_text('1\n00:00:00,000 --> 00:00:01,000\nhi\n')
    (v / 'meta.json').write_text(json.dumps({
        'id': 'VID1', 'title': 'T', 'channel': 'C', 'duration': 600,
        'upload_date': '20260101', 'webpage_url': 'https://youtube.com/watch?v=VID1',
        'last_used_at': '2000-01-01T00:00:00'}))
    # Pre-write toc.json so LLM is never called
    (v / 'toc.json').write_text(json.dumps({'sections': [
        {'path': '1', 'title': 'Intro', 'start': 0, 'end': 300},
        {'path': '2', 'title': 'Main', 'start': 300, 'end': 600},
    ]}))

    secs = generate_toc('VID1', root)
    assert len(secs) == 2
    assert secs[0]['title'] == 'Intro'
    assert secs[1]['title'] == 'Main'
print('ok')

In [ ]:
# Test 8: yttoc_toc outputs header + formatted section lines
with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'VID2'; v.mkdir()
    (v / 'captions.en.srt').write_text('1\n00:00:00,000 --> 00:00:01,000\nhi\n')
    (v / 'meta.json').write_text(json.dumps({
        'id': 'VID2', 'title': 'Test Video', 'channel': 'Ch', 'duration': 900,
        'upload_date': '20260101', 'webpage_url': 'https://youtube.com/watch?v=VID2',
        'last_used_at': '2000-01-01T00:00:00'}))
    (v / 'toc.json').write_text(json.dumps({'sections': [
        {'path': '1', 'title': 'Intro', 'start': 0, 'end': 300},
        {'path': '2', 'title': 'Main', 'start': 300, 'end': 900},
    ]}))

    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        yttoc_toc('VID2', root=str(root))
    out = buf.getvalue()

    assert '# Test Video' in out
    assert '1. Intro 0:00-5:00' in out
    assert '2. Main 5:00-15:00' in out
    assert '(5:00)' in out  # span duration
    assert '&t=300' in out  # deep link
print('ok')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()